In [2]:
#| default_exp mass.massurdf_module

In [3]:
#| export

In [ ]:
import os
import pandas as pd
from measurements_heightsC import get_measurements, get_heights

# === URDF Generation ===
def create_urdf(Ansur, inputheight):
    # if this is still raw ANSUR data, compute measurements/heights
    if 'a1' not in Ansur.columns:
        measurements = get_measurements(Ansur, inputheight).iloc[0]
        heights      = get_heights(Ansur, inputheight).iloc[0]
    else:
        measurements = Ansur.iloc[0]
        heights      = get_heights(Ansur, inputheight).iloc[0]

    # joint definitions: name → (type, axis, has_limit)
    movable = {
        # elbows
        "upper_arm_left_to_forearm":  ("revolute", (0,0,1), True),
        "upper_arm_right_to_forearm": ("revolute", (0,0,1), True),
        # knees
        "upper_leg_left_to_lower_leg":  ("revolute", (0,0,1), True),
        "upper_leg_right_to_lower_leg": ("revolute", (0,0,1), True),
    }

    def link_block(name):
        return f'''
  <link name="{name}">
    <inertial>
      <origin xyz="0 0 0" rpy="0 0 0"/>
      <mass value="0.0"/>
      <inertia 
        ixx="0.0" ixy="0.0" ixz="0.0"
        iyy="0.0" iyz="0.0"
        izz="0.0"/>
    </inertial>
    <visual>
      <geometry>
        <mesh filename="{name}.stl" scale="1 1 1"/>
      </geometry>
    </visual>
  </link>'''

    def joint_block(name, parent, child, xyz, rpy="0 0 0"):
        # choose type/axis
        jtype, axis, has_limit = movable.get(name, ("fixed", (0,0,0), False))
        axis_str = " ".join(map(str,axis))
        limit_str = ''
        if has_limit:
            limit_str = '    <limit effort="1" lower="-1.57" upper="1.57" velocity="1"/>\n'
        return f'''
  <joint name="{name}" type="{jtype}">
    <parent link="{parent}"/>
    <child  link="{child}"/>
    <origin xyz="{' '.join(map(str,xyz))}" rpy="{rpy}"/>
    <axis  xyz="{axis_str}"/>
{limit_str}  </joint>'''

    # compute all the transforms exactly as before
    tb2m = (0, 0, measurements["h3"])
    tm2t = (0, 0, measurements["h2"])
    tt2n = (0, 0, measurements["h1"] + measurements["a8"])
    n2h   = (0, 0, 0)

    # arms
    tm2uaL = (0, -(measurements["a2"]+measurements["r1"]), measurements["h2"]+measurements["h1"]-measurements["h8"])
    ua2faL = (0,0,-measurements["h10"])
    fa2hL  = (0,0,-measurements["h9"]*2)

    tm2uaR = (0,  (measurements["a2"]+measurements["r1"]), measurements["h2"]+measurements["h1"]-measurements["h8"])
    ua2faR = (0,0,-measurements["h10"])
    fa2hR  = (0,0,-measurements["h9"]*2)

    # legs
    tb2ulL = (0, -(measurements["hb"]/2-measurements["r1thigh"]), -measurements["h4"])
    ul2llL = (0,0,-measurements["h6"])
    ll2fL  = ((measurements["h7"]/2)-measurements["r2shank"], 0, -2*measurements["a5"])

    tb2ulR = (0,  (measurements["hb"]/2-measurements["r1thigh"]), -measurements["h4"])
    ul2llR = (0,0,-measurements["h6"])
    ll2fR  = ((measurements["h7"]/2)-measurements["r2shank"], 0, -2*measurements["a5"])

    # start building
    urdf = '<?xml version="1.0"?>\n<robot name="simple_human">\n'

    # all links
    for ln in ["torso_bottom","torso_mid","torso_top","neck","head",
               "upper_arm_left","forearm_left","hand_left",
               "upper_arm_right","forearm_right","hand_right",
               "upper_leg_left","lower_leg_left","foot_left",
               "upper_leg_right","lower_leg_right","foot_right"]:
        urdf += link_block(ln)

    # core joints
    urdf += joint_block("torso_bottom_to_mid","torso_bottom","torso_mid",tb2m)
    urdf += joint_block("torso_mid_to_top","torso_mid","torso_top",tm2t)
    urdf += joint_block("torso_top_to_neck","torso_top","neck",tt2n)
    urdf += joint_block("neck_to_head","neck","head",n2h)

    # left arm
    urdf += joint_block("torso_mid_to_upper_arm_left","torso_mid","upper_arm_left",tm2uaL, rpy="0 0 1.57")
    urdf += joint_block("upper_arm_left_to_forearm","upper_arm_left","forearm_left",ua2faL)
    urdf += joint_block("forearm_to_hand","forearm_left","hand_left",fa2hL)

    # right arm
    urdf += joint_block("torso_mid_to_upper_arm_right","torso_mid","upper_arm_right",tm2uaR, rpy="0 0 -1.57")
    urdf += joint_block("upper_arm_right_to_forearm","upper_arm_right","forearm_right",ua2faR)
    urdf += joint_block("forearm_to_hand_right","forearm_right","hand_right",fa2hR)

    # left leg
    urdf += joint_block("torso_bottom_to_upper_leg_left","torso_bottom","upper_leg_left",tb2ulL)
    urdf += joint_block("upper_leg_left_to_lower_leg","upper_leg_left","lower_leg_left",ul2llL)
    urdf += joint_block("lower_leg_left_to_foot","lower_leg_left","foot_left",ll2fL, rpy="0 0 1.57")

    # right leg
    urdf += joint_block("torso_bottom_to_upper_leg_right","torso_bottom","upper_leg_right",tb2ulR)
    urdf += joint_block("upper_leg_right_to_lower_leg","upper_leg_right","lower_leg_right",ul2llR)
    urdf += joint_block("lower_leg_right_to_foot","lower_leg_right","foot_right",ll2fR, rpy="0 0 1.57")

    urdf += '\n</robot>'

    # save next to your mesh folder
    script_dir  = os.path.dirname(os.path.abspath(__file__))
    output_dir  = os.path.join(script_dir, "model_output")
    os.makedirs(output_dir, exist_ok=True)
    urdf_path   = os.path.join(output_dir, "simple_human.urdf")
    with open(urdf_path, "w") as f:
        f.write(urdf)

    print(f"✅ URDF file successfully generated at: {urdf_path}")


a
